# BOSQYN IA — Entrenamiento de audio con TensorFlow / Keras

**Entrada:** `entrenamiento.zip` con 7 carpetas WAV. **Salida:** `bosqyn_tensorflow.keras` y `bosqyn_tensorflow_config.json`.

Este cuaderno **entrena realmente en Google Colab**. Usa división por grabación original para reducir filtración entre entrenamiento y prueba. No garantiza una precisión específica. Si los audios tienen pocas fuentes independientes, se necesitarán más grabaciones.


In [ ]:
%pip -q install tensorflow librosa soundfile scikit-learn matplotlib


## 1. Subir `entrenamiento.zip`

Ejecuta la celda y selecciona el ZIP original. Se copiará a `/content/entrenamiento.zip`.


In [ ]:
from google.colab import files
uploaded = files.upload()
import os
ZIP_PATH = '/content/entrenamiento.zip' if os.path.exists('/content/entrenamiento.zip') else next((f'/content/{f}' for f in uploaded if f.endswith('.zip')), None)
assert ZIP_PATH and os.path.isfile(ZIP_PATH), 'Sube entrenamiento.zip'
print('ZIP:', ZIP_PATH)


## 2. Validar WAV y extraer espectrogramas Mel

Cada WAV se convierte a 16 kHz mono, duración 3 s. El grupo de procedencia se obtiene del nombre original (por ejemplo, `freesound_135821`).


In [ ]:
import os, io, re, json, zipfile, pathlib
import numpy as np, librosa, soundfile as sf, tensorflow as tf
from collections import Counter
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.utils.class_weight import compute_class_weight
SEED=42
np.random.seed(SEED); tf.random.set_seed(SEED)
SR=16000; SECONDS=3; N_MELS=64; HOP=256; FFT=1024; TARGET_FRAMES=188
with zipfile.ZipFile(ZIP_PATH) as z:
    names=[p for p in z.namelist() if p.lower().endswith('.wav') and not p.startswith('__MACOSX')]
    classes=sorted({p.split('/')[0] for p in names if '/' in p})
    assert len(classes)==7, f'Se esperaban 7 categorías; encontradas: {classes}'
    X=[]; Y=[]; groups=[]; errors=[]
    for i,p in enumerate(names):
        try:
            audio, sr=sf.read(io.BytesIO(z.read(p)),dtype='float32')
            if audio.ndim==2: audio=audio.mean(axis=1)
            if len(audio)==0 or not np.isfinite(audio).all(): raise ValueError('Audio vacío o no finito')
            if sr!=SR: audio=librosa.resample(audio,orig_sr=sr,target_sr=SR)
            audio=librosa.util.fix_length(audio,size=SR*SECONDS)
            mel=librosa.feature.melspectrogram(y=audio,sr=SR,n_fft=FFT,hop_length=HOP,n_mels=N_MELS,fmax=7600)
            mel=librosa.power_to_db(mel,ref=1.0)
            mel=librosa.util.fix_length(mel,size=TARGET_FRAMES,axis=1)
            X.append(mel.astype('float32')); Y.append(classes.index(p.split('/')[0]))
            stem=pathlib.Path(p).stem
            m=re.search(r'(?:freesound|commons|zenodo|youtube)_([^_]+)',stem,re.I)
            group=p.split('/')[0]+'/'+(m.group(0).lower() if m else stem)
            groups.append(group)
        except Exception as e: errors.append((p,str(e)))
X=np.asarray(X,dtype='float32'); Y=np.asarray(Y,dtype='int32'); groups=np.asarray(groups)
print('Clases:', classes)
print('Audios válidos:',len(Y),'Grupos independientes aproximados:',len(set(groups)),'Errores:',len(errors))
print('Por clase:',{c:int(sum(Y==i)) for i,c in enumerate(classes)})
assert len(Y)>0


## 3. Separar por grabación original

70 % de grupos para entrenar, 15 % validación, 15 % prueba por categoría. La estimación de procedencia se basa en los nombres de archivo: revisa que identifiquen correctamente las fuentes.


In [ ]:
rng=np.random.default_rng(SEED)
split=np.zeros(len(Y),dtype=np.int8)
for c in range(len(classes)):
    indices=np.where(Y==c)[0]; uniq=np.unique(groups[indices]); rng.shuffle(uniq)
    if len(uniq)<4: raise ValueError(f'Insuficientes grabaciones independientes para {classes[c]}: {len(uniq)}')
    nt=max(1,round(len(uniq)*.15)); nv=max(1,round(len(uniq)*.15))
    mapping={g:(2 if j<nt else 1 if j<nt+nv else 0) for j,g in enumerate(uniq)}
    for ix in indices: split[ix]=mapping[groups[ix]]
train=np.where(split==0)[0]; val=np.where(split==1)[0]; test=np.where(split==2)[0]
mean=float(X[train].mean()); std=float(X[train].std()+1e-6)
X=np.clip((X-mean)/std,-5,5)[...,None]
print('Train',len(train),'Val',len(val),'Test',len(test))


## 4. Entrenar CNN TensorFlow

La red incorpora normalización, dropout, ajuste de tasa de aprendizaje y parada temprana. Si la precisión sigue baja, el problema puede ser la diversidad de fuentes, no solo la arquitectura.


In [ ]:
tf.keras.backend.clear_session()
model=tf.keras.Sequential([
 tf.keras.layers.Input(shape=(N_MELS,TARGET_FRAMES,1)),
 tf.keras.layers.Conv2D(32,3,padding='same',activation='relu'),tf.keras.layers.BatchNormalization(),tf.keras.layers.MaxPooling2D(),
 tf.keras.layers.Conv2D(64,3,padding='same',activation='relu'),tf.keras.layers.BatchNormalization(),tf.keras.layers.MaxPooling2D(),
 tf.keras.layers.Conv2D(128,3,padding='same',activation='relu'),tf.keras.layers.BatchNormalization(),tf.keras.layers.MaxPooling2D(),
 tf.keras.layers.Conv2D(128,3,padding='same',activation='relu'),tf.keras.layers.GlobalAveragePooling2D(),
 tf.keras.layers.Dense(128,activation='relu'),tf.keras.layers.Dropout(.45),
 tf.keras.layers.Dense(len(classes),activation='softmax')
])
model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=.0005),loss='sparse_categorical_crossentropy',metrics=['accuracy'])
weights=compute_class_weight('balanced',classes=np.arange(len(classes)),y=Y[train]); class_weight={i:float(w) for i,w in enumerate(weights)}
callbacks=[tf.keras.callbacks.EarlyStopping(monitor='val_loss',patience=9,restore_best_weights=True),tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss',patience=4,factor=.5,min_lr=1e-6)]
history=model.fit(X[train],Y[train],validation_data=(X[val],Y[val]),epochs=50,batch_size=32,callbacks=callbacks,class_weight=class_weight,verbose=2)


## 5. Evaluación y exportación `.keras`


In [ ]:
probs=model.predict(X[test],verbose=0); preds=probs.argmax(axis=1)
print(classification_report(Y[test],preds,labels=np.arange(len(classes)),target_names=classes,zero_division=0))
print('Matriz de confusión:'); print(confusion_matrix(Y[test],preds,labels=np.arange(len(classes))))
accuracy=float(np.mean(preds==Y[test])); print('Precisión en prueba:',round(accuracy*100,2),'%')
model_path='/content/bosqyn_tensorflow.keras'; model.save(model_path)
config={'classes':classes,'sample_rate':SR,'clip_seconds':SECONDS,'n_mels':N_MELS,'n_fft':FFT,'hop_length':HOP,'frames':TARGET_FRAMES,'fmax':7600,'mean':mean,'std':std,'backend':'TensorFlow','test_accuracy':accuracy,'train_samples':len(train),'val_samples':len(val),'test_samples':len(test),'approximate_recording_groups':len(set(groups)),'invalid_files':errors}
with open('/content/bosqyn_tensorflow_config.json','w',encoding='utf8') as f:json.dump(config,f,indent=2,ensure_ascii=False)
# Verificar que el archivo .keras se abre con TensorFlow
restored=tf.keras.models.load_model(model_path)
assert restored.output_shape[-1]==7
print('Modelo guardado y verificado:',model_path)
from google.colab import files
files.download(model_path)
files.download('/content/bosqyn_tensorflow_config.json')
